**INSTALL DEPENDENCIES**

In [13]:
!pip install deep-translator tqdm

**IMPORTS**

In [14]:
import os
import json
import time
from deep_translator import GoogleTranslator
from concurrent.futures import ThreadPoolExecutor

**MOUNT DRIVE**

In [17]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


**PATH SETUP**

In [18]:
INPUT_JSON = "/content/drive/MyDrive/BanglaVision/bd_captions_en.json"
OUTPUT_FOLDER = "/content/drive/MyDrive/BanglaVision/bd_translated_batches"

BATCH_SIZE = 1000
NUM_WORKERS = 5

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Output folder ready:", OUTPUT_FOLDER)

Output folder ready: /content/drive/MyDrive/BanglaVision/bd_translated_batches


**VERIFY FILES**

In [19]:
print("Files inside BanglaVision:")
print(os.listdir("/content/drive/MyDrive/BanglaVision"))

Files inside BanglaVision:
['bd_images', 'create_bd_image_dataset.ipynb', 'bd_captions_en.json', 'bd_image_caption_generation.ipynb', 'Untitled0.ipynb', 'bd_translated_batches']


**LOAD DATA**

In [20]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total items:", len(data))

Total items: 11593


**TRANSLATE FUNCTION**

In [21]:
def safe_translate(text, retries=2):
    for _ in range(retries):
        try:
            result = GoogleTranslator(source='en', target='bn').translate(text)

            if result and result.strip().lower() != text.strip().lower():
                return result

        except Exception:
            pass

        time.sleep(1)

    return "__FAILED__"

**PROCESS CHUNK PARALLEL**

In [22]:
def process_chunk(chunk):
    results = []

    for item in chunk:
        caption_en = item["caption_en"]
        image_path = item["image"]

        caption_bn = safe_translate(caption_en)

        results.append({
            "image": image_path,
            "caption_en": caption_en,
            "caption_bn": caption_bn
        })

    return results

**MAIN TRANSLATION LOOP**

In [23]:
for start in range(0, len(data), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(data))
    output_file = os.path.join(OUTPUT_FOLDER, f"batch_{start}_{end}.json")

    if os.path.exists(output_file):
        print(f"Skipping {start} to {end}")
        continue

    print(f"Processing {start} to {end}")

    batch_data = data[start:end]

    chunk_size = len(batch_data) // NUM_WORKERS + 1
    chunks = [batch_data[i:i + chunk_size] for i in range(0, len(batch_data), chunk_size)]

    results = []

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for res in executor.map(process_chunk, chunks):
            results.extend(res)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    print(f"Saved batch {start} to {end}")

    time.sleep(1)

print("Translation process completed!")

Processing 0 to 1000
Saved batch 0 to 1000
Processing 1000 to 2000
Saved batch 1000 to 2000
Processing 2000 to 3000
Saved batch 2000 to 3000
Processing 3000 to 4000
Saved batch 3000 to 4000
Processing 4000 to 5000
Saved batch 4000 to 5000
Processing 5000 to 6000
Saved batch 5000 to 6000
Processing 6000 to 7000
Saved batch 6000 to 7000
Processing 7000 to 8000
Saved batch 7000 to 8000
Processing 8000 to 9000
Saved batch 8000 to 9000
Processing 9000 to 10000
Saved batch 9000 to 10000
Processing 10000 to 11000
Saved batch 10000 to 11000
Processing 11000 to 11593
Saved batch 11000 to 11593
Translation process completed!


**MERGE ALL BATCHES**

In [24]:
import os
import json

OUTPUT_FOLDER = "/content/drive/MyDrive/BanglaVision/bd_translated_batches"

all_data = []

files = sorted(os.listdir(OUTPUT_FOLDER))

print("Total batch files:", len(files))

for file in files:
    file_path = os.path.join(OUTPUT_FOLDER, file)

    if file.endswith(".json"):
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            all_data.extend(data)

print("Total merged samples:", len(all_data))

Total batch files: 12
Total merged samples: 11593


**CLEAN FAILED TRANSLATIONS**

In [40]:
clean_data = [x for x in all_data if x["caption_bn"] != "__FAILED__"]

print("Original:", len(all_data))
print("Clean:", len(clean_data))

Original: 11593
Clean: 11593


**EXTRACT FAILED SAMPLES**

In [41]:
failed_data = [x for x in all_data if x["caption_bn"] == "__FAILED__"]

print("Failed samples:", len(failed_data))

Failed samples: 0


**SAFE TRANSLATION**

In [28]:
from deep_translator import GoogleTranslator
import time
from tqdm import tqdm

def safe_translate_slow(text, retries=3):
    for _ in range(retries):
        try:
            result = GoogleTranslator(source='en', target='bn').translate(text)

            if result and result.strip():
                return result

        except:
            pass

        time.sleep(2)

    return "__FAILED__"

**RETRANSLATE FAILED**

In [29]:
recovered = []

for item in tqdm(failed_data):

    caption_en = item["caption_en"]

    new_bn = safe_translate_slow(caption_en)

    item["caption_bn"] = new_bn

    recovered.append(item)

100%|██████████| 3703/3703 [13:59<00:00,  4.41it/s]


**MERGE BACK INTO DATASET**

In [30]:
recovered_map = {x["image"]: x for x in recovered}

final_data = []

for item in all_data:
    if item["image"] in recovered_map:
        final_data.append(recovered_map[item["image"]])
    else:
        final_data.append(item)

print("Final dataset size:", len(final_data))

Final dataset size: 11593


**FINAL CLEAN**

In [31]:
clean_final = []

for x in final_data:
    if x["caption_bn"] == "__FAILED__":
        x["caption_bn"] = x["caption_en"]
    clean_final.append(x)

print("Final usable samples:", len(clean_final))

Final usable samples: 11593


**SAVE FINAL DATASET**

In [39]:
FINAL_OUTPUT = "/content/drive/MyDrive/BanglaVision/bd_captions_bn_final.json"

with open(FINAL_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(clean_final, f, indent=2, ensure_ascii=False)

print("Final dataset saved:", FINAL_OUTPUT)

Final dataset saved: /content/drive/MyDrive/BanglaVision/bd_captions_bn_final.json
